# AnnData, `.h5ad`, and pseudobulk for this challenge

This notebook is for **competitors** who want to understand:

1. **Single-cell RNA-seq** in plain language — why we work at cell resolution (with a pointer to the [10x Genomics guide](https://www.10xgenomics.com/what-is-single-cell-rna-seq)).
2. **Genotyping** — SNP-level DNA variation per donor (`0/0`, `0/1`, `1/1`); multimodal use alongside expression (data release to follow).
3. **AnnData** and the **`.h5ad`** file format used across single-cell tooling (Scanpy, scVI, etc.).
4. How this repository builds **pseudobulk** from cell-level data for age prediction.

**Official AnnData documentation:** [anndata.readthedocs.io](https://anndata.readthedocs.io/en/latest/index.html) — the format sits between pandas and tensor-style arrays and is central to the [scverse](https://scverse.org/) ecosystem.

**Requirements:** `scanpy`, `pandas`, `numpy` (same environment as the baseline model tutorial).

**New to biology?** Read **`00_biology_genome_and_ngs_primer.ipynb`** first (genome → RNA-seq → `.h5ad`).

## Single-cell RNA sequencing (scRNA-seq)

The competition data come from **single-cell** experiments: each observation in the cell-level `.h5ad` is one cell (or one cell type within one donor after aggregation). The following overview follows the beginner-oriented framing in the **10x Genomics** learning hub — [What is single cell RNA sequencing?](https://www.10xgenomics.com/what-is-single-cell-rna-seq).

**What it measures.** Single-cell RNA sequencing measures **gene expression in individual cells** (typically thousands of genes per cell). That lets you recover **which genes are expressed in which cells**, instead of averaging over a whole tissue or blood sample.

**Why not only bulk RNA-seq?** In **bulk** RNA-seq, expression is averaged across many cells in one tube. That can hide rare populations and mixed cell states. For example, in a tumour sample, a small subset of treatment-resistant cells might be invisible in bulk data but detectable when each cell is measured separately. scRNA-seq is often used to map **heterogeneity**, **rare cell types**, and **dynamic states** that bulk methods miss.

**What researchers use it for** (examples from the same resource): building atlases of healthy vs diseased tissue, tracking developmental or immune dynamics, studying **cell–cell** communication in niches such as tumours, and defining new cell types or states with genome-wide readouts rather than only a fixed marker panel.

**Trade-offs (high level).** Single-cell experiments usually need **more depth and specialised analysis** than bulk, and workflows can be more involved — but the field standardises on shared objects (here: **AnnData** / `.h5ad`) and tools such as Scanpy so downstream steps are reproducible.

*Source: 10x Genomics, “A beginner's guide to building confidence in single cell RNA-seq” — [link](https://www.10xgenomics.com/what-is-single-cell-rna-seq).*


## Genotyping and SNP matrices

The main files in this repository are **RNA expression** (`.h5ad`). The challenge will also provide **genotypic** data: for each **donor**, the genotype at many **SNP** positions across the genome. You can use this for **multimodal** models (expression + genetics), joined on `donor_id`, when the files are released.

**What is a SNP?** A **single-nucleotide polymorphism** is a position in the genome where the population carries two (or more) alternative DNA letters (alleles), often called **reference** vs **alternate** relative to a reference genome build.

**Diploid genotypes.** Humans carry **two** copies of each autosome (one from each parent). At each biallelic SNP, a person is either **homozygous** (same allele on both copies) or **heterozygous** (one copy of each allele).

**Encoding `0/0`, `0/1`, `1/1`.** A common symbolic coding for one SNP with reference allele “0” and alternate allele “1” is:

| Code | Meaning (typical) |
|------|-------------------|
| **`0/0`** | Homozygous **reference** — both chromosomes carry the reference allele |
| **`0/1`** (or `1/0`) | **Heterozygous** — one reference, one alternate allele |
| **`1/1`** | Homozygous **alternate** — both chromosomes carry the alternate allele |

Some pipelines instead store a numeric **dosage** in `{0, 1, 2}` = number of alternate alleles; the idea is the same discrete summary per SNP.

**Matrix layout.** The release will be organized as **individuals × SNPs** (rows aligned to challenge **donors**, columns = SNP IDs). Each entry is a genotype class (or dosage). This table is **not** an AnnData object by itself — you load it with **pandas** / **NumPy** (or similar) and merge features with expression-based models on **`donor_id`** where rules allow.

**Relation to RNA.** RNA-seq measures **gene expression** (which genes are *used*). Genotyping measures **DNA sequence variation** (*which variants* a person carries). They answer different questions; combining them can improve prediction when both are available.

*Full genotypic files and documentation will be published when released; this section is only conceptual.*


## Part A — What is AnnData?

An **`AnnData`** object (`anndata.AnnData`) bundles:

| Slot | Role | Typical content |
|------|------|-----------------|
| **`X`** | Main data matrix | `n_obs × n_vars` (cells × genes), often sparse counts |
| **`obs`** | Row annotations | cell IDs, QC, `donor_id`, `celltype`, `age`, … |
| **`var`** | Column annotations | gene symbols / Ensembl IDs, optional flags |
| **`layers`** | Extra matrices | raw counts, normalized data, imputed |
| **`obsm` / `varm`** | Multi-column embeddings | PCA, UMAP coordinates |
| **`uns`** | Unstructured dict | pipeline parameters, colors, … |

**Naming:** `obs` = observations (here: **cells**, or later **donor×celltype** rows). `var` = variables (here: **genes**).

**.h5ad** is the HDF5-based on-disk format for AnnData (`read_h5ad` / `write_h5ad`). See the AnnData docs for [on-disk layout and interoperability](https://anndata.readthedocs.io/en/latest/index.html).

**Teaching point:** Inspect `adata.shape`, `adata.obs.head()`, and `adata.var_names` first whenever you open a new file.

### Visual schema

![AnnData schematic: matrix X with obs (rows) and var (columns) and linked annotation slots](../images/anndata_schema_image.svg)

Repository file: [`images/anndata_schema_image.svg`](../images/anndata_schema_image.svg) (relative to project root; from this notebook the path above is `../images/...`).

### A1 — Project path

Run from the repository root or from `notebooks/`.

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find models/train_age_model.py. Open Jupyter from the repo root or from notebooks/."
    )

# Competition data live under `data/` (bind or copy from shared scratch — see README).
DATA_DIR = PROJ_ROOT / "data"
RESULTS_DIR = PROJ_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJ_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

### A2 — Explore competition **cell-level** `.h5ad` (lightweight peek)

Full objects (~1M+ cells) are **large in RAM**. Two strategies:

1. **`backed='r'`** — keep `X` on disk while you inspect `obs` / `var` (see [AnnData backed mode](https://anndata.readthedocs.io/en/latest/generated/anndata.AnnData.html)).
2. **Pseudobulk outputs** (next sections) — much smaller; use those for quick plots.

Files are read from **`data/`** (e.g. `data/scRNA-seq_raw/train.h5ad`). Adjust `CELL_LEVEL` only if you use a custom layout.

In [ ]:
import scanpy as sc

CELL_LEVEL = DATA_DIR / "scRNA-seq_raw" / "train.h5ad"

if not CELL_LEVEL.exists():
    print("No cell-level file at:", CELL_LEVEL)
    print("Copy or bind competition data into data/ (see README), or skip to pseudobulk sections.")
else:
    ad_cell = sc.read_h5ad(CELL_LEVEL, backed="r")
    try:
        print("Shape (obs × var):", ad_cell.n_obs, "×", ad_cell.n_vars)
        print("obs columns (sample):", list(ad_cell.obs.columns[:12]), "...")
        print(ad_cell.obs[["donor_id", "celltype", "age", "_split"]].head(6))
    finally:
        if getattr(ad_cell, "isbacked", False) and ad_cell.file is not None:
            ad_cell.file.close()

**Competition columns (cell-level `obs`):** `donor_id`, `celltype` (5 coarse groups), `age` (missing on test cells in some builds), `_split` (`train` / `val` / `test`), plus QC fields — see the main `README.md`.

Each **row** is one cell; `X[i, g]` is the expression of gene `g` in cell `i`.

### A3 — Explore train **donor-aggregated** pseudobulk `.h5ad`

Use the train split donor-level matrix directly:

- `data/scRNA-seq_pseudobulk/train_pseudobulk_donor_aggregated_public.h5ad`

This matrix has one row per donor and feature columns like `ENSG...__CD4_T`, `ENSG...__CD8_T`, etc. (5 cell-type slots per gene).

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

PB_TRAIN = DATA_DIR / "scRNA-seq_pseudobulk" / "train_pseudobulk_donor_aggregated_public.h5ad"

if not PB_TRAIN.exists():
    print("No train donor-aggregated pseudobulk file at:", PB_TRAIN)
    print("Copy or bind competition data into data/ (see data/README.md).")
else:
    ad_pb = sc.read_h5ad(PB_TRAIN)
    print("Shape:", ad_pb.n_obs, "donors ×", ad_pb.n_vars, "features")
    print("obs columns:", list(ad_pb.obs.columns))

    obs_cols = [c for c in ("donor_id", "age", "_split") if c in ad_pb.obs.columns]
    display(ad_pb.obs[obs_cols].head(10))

    # Features live in X/var as gene__celltype columns
    g0 = ad_pb.var_names[0]
    print(f"Example feature {g0!r} — first donors:")
    Xcol = ad_pb[:10, g0].X
    if hasattr(Xcol, "toarray"):
        Xcol = Xcol.toarray()
    values = np.asarray(Xcol).ravel()

    demo = ad_pb.obs[obs_cols].head(10).copy() if obs_cols else pd.DataFrame(index=ad_pb.obs.index[:10])
    demo[g0] = values
    display(demo)

### A4 — **Donor-aggregated** matrix (what the baseline model reads)

`scRNA-seq_pseudobulk/train_pseudobulk_donor_aggregated_public.h5ad` (train split) has **one row per donor** and **one column per (gene, cell-type slot)** with names like `ENSG...__CD4_T`.

So each gene contributes **5 numbers** per donor (CD4 T, CD8 T, NK, B cells, monocytes). Total columns ≈ `5 × n_genes`. Use `val_` / `test_` files for those splits.

In [ ]:
PB_AGG = DATA_DIR / "scRNA-seq_pseudobulk" / "train_pseudobulk_donor_aggregated_public.h5ad"

if not PB_AGG.exists():
    print("No donor-aggregated file at:", PB_AGG)
else:
    ad_agg = sc.read_h5ad(PB_AGG)
    print("Shape:", ad_agg.n_obs, "donors ×", ad_agg.n_vars, "features")
    print("Example var_names:", list(ad_agg.var_names[:6]))
    display(ad_agg.obs[["donor_id", "age", "_split"]].head())

## Part B — Pseudobulk method (this repository)

### Step 1 — Sum counts per **(donor, cell type)**

From cell-level `AnnData`, for each cell type group we compute:

$$ \text{pseudobulk}_{d,c,g} = \sum_{\text{cells } i \text{ with donor } d,\, \text{type } c} X_{i,g} $$

Implementation: `h5ad_to_pseudobulk()` in `data_prep/h5ad_to_pseudobulk.py` loops over cell types, masks cells, then sums each donor’s rows in `X` (or a chosen `layer`). Donor metadata is attached from the first matching cell; **`n_cells`** stores how many cells were summed per donor and type.

Per-cell-type objects are written with the input prefix (e.g. `train_CD4_T.h5ad` when input is `train.h5ad`).

### Step 2 — Stack all types → **long-form** `(donor, celltype) × genes`

`scanpy.concat(..., axis=0)` stacks the per-type `AnnData` objects so each row is still one donor in one cell type.

### Step 3 — **Donor-level wide matrix** for ML

`aggregate_to_donor_level()` reshapes to **one row per donor** and fills blocks of columns in fixed order: CD4 T, CD8 T, NK, B cells, monocytes — so feature `GENE__CD4_T` is the pseudobulk count of that gene in CD4 T cells for that donor, etc.

**Why pseudobulk?** It collapses millions of cells into **interpretable donor-level features**, reduces noise from individual dropout events, and matches common practice when the prediction target is **donor metadata** (here: age) rather than single-cell labels.

**Caveat:** Summing raw counts assumes counts are comparable within cell type; depth differences across donors still matter (competitors may normalise or use offsets — the baseline applies `log1p` before the Random Forest).

### B1 — **Minimal synthetic example** (always runs)

We build a toy `AnnData` with 2 donors × 5 cell types × a few genes and call the **same functions** as the pipeline so you see the shapes change: cells → pseudobulk by type → concatenated → donor-aggregated.

In [ ]:
import sys

sys.path.insert(0, str(PROJ_ROOT / "data_prep"))
import h5ad_to_pseudobulk as pb  # noqa: E402

np.random.seed(0)
cts = ["CD4 T", "CD8 T", "NK", "B cells", "monocytes"]
genes = ["GENE_A", "GENE_B", "GENE_C"]
rows = []
X_rows = []
# Two donors; each donor has 3 cells per cell type (30 rows total)
for donor in ["101", "102"]:
    base = 20 if donor == "101" else 35
    for ct in cts:
        for _cell in range(3):
            rows.append({"donor_id": donor, "celltype": ct, "age": float(base), "_split": "train"})
            noise = np.random.poisson(1, size=len(genes))
            signal = np.array([5, 2, 1]) * (cts.index(ct) + 1)
            X_rows.append(signal + noise)

obs_toy = pd.DataFrame(rows)
X_toy = np.asarray(X_rows, dtype=np.float64)
var_toy = pd.DataFrame(index=genes)
adata_toy = sc.AnnData(X=X_toy, obs=obs_toy, var=var_toy)

print("Cell-level:", adata_toy.n_obs, "cells ×", adata_toy.n_vars, "genes")
by_ct = pb.h5ad_to_pseudobulk(adata_toy, donor_col="donor_id", celltype_col="celltype")
for ct, sub in by_ct.items():
    print(f"  {ct}: {sub.n_obs} donors × {sub.n_vars} genes (sums), n_cells col =", sub.obs["n_cells"].tolist())

ad_stack = sc.concat(list(by_ct.values()), axis=0, join="inner")
print("Stacked:", ad_stack.n_obs, "(donor×type) rows ×", ad_stack.n_vars, "genes")

ad_wide = pb.aggregate_to_donor_level(ad_stack, donor_col="donor_id", celltype_col="celltype")
print("Donor-aggregated:", ad_wide.n_obs, "donors ×", ad_wide.n_vars, "features")
print("Feature names (3 genes × 5 types = 15):", list(ad_wide.var_names))

### B2 — Reproduce pipeline command (optional)

From the project root, using the public split files under `data/scRNA-seq_raw/`:

```bash
python data_prep/h5ad_to_pseudobulk.py data/scRNA-seq_raw/train.h5ad -o results/pseudobulk_train
```

Or via your Apptainer wrapper / SLURM script pointing at that Python module.

***

**Further reading:** [AnnData API](https://anndata.readthedocs.io/en/latest/api.html), project `README.md` (Pseudobulk section), and `data_prep/h5ad_to_pseudobulk.py` source.